In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import ipywidgets as widgets
import os
from IPython.display import display, clear_output

%matplotlib inline

In [ ]:

# Load all .dat files in the current directory

# Change directory to ../data before loading files
os.chdir('../data')

# Get list of .dat files
dat_files = [f for f in os.listdir() if f.endswith('.dat')]

# Create dropdown widget for file selection
def load_data(file):
    df = pd.read_csv(file, header=0, names=["timestep", "robot_id", "x", "y", "yaw"])
    return df

file_selector = widgets.Dropdown(
    options=dat_files,
    description='Select file:',
    disabled=False,
)
display(file_selector)
print("Select a file to load data.")

df = load_data(dat_files[-1])

# Update the file_name based on the selection
def on_file_change(change):
    global file_name, df
    file_name = change['new']

    df = load_data(file_name)

file_selector.observe(on_file_change, names='value')


def plot_all_robot_paths():
    clear_output(wait=True)
    display(file_selector)
    plt.figure(figsize=(12, 8))
    df['x'] = df['x'].astype(float)
    df['y'] = df['y'].astype(float)
    df['robot_id'] = df['robot_id'].astype(str)
    for robot in df['robot_id'].unique():
        robot_data = df[df['robot_id'] == robot]
        plt.plot(robot_data['x'], robot_data['y'], label=robot)
    plt.title('Paths of All Robots')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.grid(True)
    plt.axis('equal')
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
    plt.show()

file_selector.observe(lambda change: plot_all_robot_paths(), names='value')
plot_all_robot_paths()


In [ ]:
# Create dropdown for robot ID selection
robot_ids = df['robot_id'].unique()
robot_selector = widgets.Dropdown(
    options=robot_ids,
    description='Robot ID:',
    disabled=False,
)
display(robot_selector)


# Update the robot_id based on the selection
def on_robot_change(change):
    global robot_id
    robot_id = change['new']
robot_selector.observe(on_robot_change, names='value')

def plot_robot_path(change):
    clear_output(wait=True)
    display(robot_selector)
    plt.figure(figsize=(10, 6))
    robot_data = df[df['robot_id'] == robot_selector.value]
    robot_data['x'] = robot_data['x'].astype(float)
    robot_data['y'] = robot_data['y'].astype(float)
    robot_data['timestep'] = robot_data['timestep'].astype(int)  # Ensure timestep is numeric for coloring
    plt.scatter(robot_data['x'], robot_data['y'], c=robot_data['timestep'], cmap='hot', s=10)
    plt.plot(robot_data['x'].iloc[-1], robot_data['y'].iloc[-1], 'k*', markersize=15, label='Last Point')
    plt.colorbar(label='Timestep')
    plt.title(f'Path of Robot {robot_selector.value}')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.grid(True)
    plt.axis('equal')
    plt.show()

robot_selector.observe(plot_robot_path, names='value')

In [ ]:
robot_data = df[df['robot_id'] == robot_selector.value]
robot_data

In [ ]:
# Create an IntSlider for timestep selection
timestep_slider = widgets.IntSlider(
    value=df['timestep'].min(),
    min=df['timestep'].min(),
    max=df['timestep'].max(),
    step=1,
    description='Timestep:',
    continuous_update=False
)
display(timestep_slider)

def plot_robots_at_timestep(change):
    clear_output(wait=True)
    display(timestep_slider)
    timestep = timestep_slider.value
    timestep_data = df[df['timestep'] == timestep].dropna(subset=['x', 'y', 'yaw'])
    
    plt.figure(figsize=(12, 8))
    plt.quiver(
        timestep_data['x'], timestep_data['y'], 
        np.cos(timestep_data['yaw']), np.sin(timestep_data['yaw']), 
        angles='xy', scale_units='xy', scale=5, color='blue'
    )
    plt.title(f'Robot Positions and Orientations at Timestep {timestep}')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.grid(True)
    plt.axis('equal')
    plt.show()

timestep_slider.observe(plot_robots_at_timestep, names='value')
plot_robots_at_timestep(None)